# Профилирование источников модельного риска на этапе разработки

Ноутбук формирует таблицу `arnsdpsbx_t_team_oam_sva_2.pri_model_lib_source_profile`.

В неё входят четыре вида записей:

- `TABLE_SUMMARY` — размер актуального периметра и оценка дублей ключа;
- `COLUMN_PROFILE` — заполненность, заглушки, уникальность, диапазоны, распознаваемость чисел и дат;
- `TOP_VALUE` — частые значения статусов, типов, флагов, ролей выборок и названий метрик;
- `ROW_SAMPLE` — компактные показательные строки для проверки связей между сущностями.

Пути, ссылки, комментарии и свободные тексты в `ROW_SAMPLE` хешируются. Колонки с ФИО и пользовательскими идентификаторами исключаются.


In [ ]:
import os
import re
import sys
from typing import Dict, List

os.environ["SPARK_MAJOR_VERSION"] = "3.5.1"
os.environ["SPARK_HOME"] = "/usr/sdp/current/spark3.5.1-client/"
os.environ["PYSPARK_DRIVER"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

sys.path.insert(0, "/usr/sdp/current/spark3.5.1-client/python/")
sys.path.insert(
    0,
    "/usr/sdp/current/spark3.5.1-client/python/lib/py4j-0.10.9.7-src.zip",
)

from pyspark import SparkConf, StorageLevel
from pyspark.sql import DataFrame, SparkSession, functions as F, types as T

SRC_DB = "prx_pri_custom_ris_l_library_custom_risk_model_library"
OUT_DB = "arnsdpsbx_t_team_oam_sva_2"
OUT_TABLE = "pri_model_lib_source_profile"
OUT_FULL_NAME = f"{OUT_DB}.{OUT_TABLE}"

MAX_PROFILE_ROWS = 100_000
ROW_SAMPLES_PER_TABLE = 20
TOP_VALUES_PER_COLUMN = 20
SEED = 42

conf = (
    SparkConf()
    .setAppName("MODEL_LIBRARY_SOURCE_PROFILE")
    .setMaster("yarn")
    .set("spark.executor.cores", "2")
    .set("spark.executor.memory", "6g")
    .set("spark.executor.memoryOverhead", "1g")
    .set("spark.driver.memory", "6g")
    .set("spark.driver.maxResultSize", "1g")
    .set("spark.shuffle.service.enabled", "true")
    .set("spark.dynamicAllocation.enabled", "true")
    .set("spark.dynamicAllocation.initialExecutors", "3")
    .set("spark.dynamicAllocation.maxExecutors", "12")
    .set("spark.dynamicAllocation.shuffleTracking.enabled", "true")
    .set("spark.sql.parquet.compression.codec", "snappy")
    .set("spark.sql.session.timeZone", "Europe/Moscow")
)

spark = SparkSession.builder.config(conf=conf).enableHiveSupport().getOrCreate()
print(f"Output: {OUT_FULL_NAME}")


In [ ]:
# Только поля, которые нужны для оценки применимости дополнительных контролей.
# Если поле или таблица отсутствуют физически, это будет записано в профиль,
# а выполнение продолжится.

TABLES: Dict[str, Dict[str, object]] = {
    "t_model": {
        "key": "model_sid",
        "columns": [
            "model_sid", "model_name", "model_stts_name", "model_type_name",
            "model_subtype_name", "model_ml_task_name", "model_rsk_flag",
            "model_rsk_type_name", "model_rsk_sgmnt_name", "model_strat_flag",
            "model_ifrs_calc_flag", "model_conf_ctgry_name",
            "model_conf_ctgry_neg_flag", "model_dev_dprtmt_name",
            "model_proc_type_name", "model_crtn_dttm",
        ],
        "top": [
            "model_stts_name", "model_type_name", "model_subtype_name",
            "model_ml_task_name", "model_rsk_flag", "model_rsk_type_name",
            "model_rsk_sgmnt_name", "model_strat_flag", "model_ifrs_calc_flag",
            "model_conf_ctgry_name", "model_conf_ctgry_neg_flag",
            "model_proc_type_name",
        ],
        "rows": [
            "model_sid", "model_stts_name", "model_type_name",
            "model_subtype_name", "model_ml_task_name", "model_rsk_flag",
            "model_rsk_type_name", "model_rsk_sgmnt_name",
            "model_ifrs_calc_flag", "model_conf_ctgry_name",
        ],
    },
    "t_model_ver": {
        "key": "model_ver_sid",
        "columns": [
            "model_ver_sid", "model_sid", "model_ver_num", "model_ver_name",
            "model_ver_stts_name", "model_ver_crtn_dttm",
            "model_ver_signfcnt_ctgry_code", "model_ver_signfcnt_ctgry_descr_txt",
            "model_ver_signfcnt_ctgry_dttm", "model_ver_signfcnt_lvl_name",
            "model_ver_supervision_lvl_name", "model_ver_irb_flag",
            "model_ver_owner_dprtmt_name", "model_ver_dev_dprtmt_name",
            "model_ver_method_name", "model_ver_tgt_txt",
            "model_ver_task_type_name", "model_ver_data_type_name",
            "model_ver_role_array", "model_ver_prod_name",
            "model_ver_appl_claim_crpnd_txt", "model_ver_initr_jira_task_num",
            "model_ver_dev_src_name", "model_ver_dev_sys_name",
            "model_ver_dev_instr_name", "model_ver_dev_start_plan_dttm",
            "model_ver_dev_start_fact_dttm", "model_ver_dev_end_plan_dttm",
            "model_ver_dev_end_fact_dttm", "model_ver_dev_not_sample_data_flag",
            "model_ver_data_mart_link_txt", "model_ver_mlstorage_proj_name",
            "model_ver_mlstorage_dataset_link_txt",
            "model_ver_mlstorage_branch_link_txt", "model_ver_dev_report_sid",
            "model_ver_dev_report_conf_ctgry_name",
            "model_ver_dev_report_tmplt_sid",
            "model_ver_dev_report_proj_zone_name",
            "model_ver_repstry_link_txt",
            "model_ver_repstry_non_link_reason_name",
            "model_ver_repstry_non_link_reason_cmnt_txt",
            "model_ver_repstry_commit_sid", "model_ver_repstry_branch_name",
            "model_ver_repstry_branch_main_name",
            "model_ver_param_team_area_link_txt", "model_ver_pipeline_name",
            "model_ver_proj_feature_flag", "model_ver_proj_feature_sid",
            "model_ver_prevalid_flag", "model_ver_prevalid_proj_sid",
            "model_ver_prevalid_proj_link_txt",
            "model_ver_prevalid_report_link_txt",
            "model_ver_prevalid_report_link_sid",
            "model_ver_prevalid_report_file_sid",
            "model_ver_prevalid_rslt_name",
            "model_ver_prevalid_rslt_manual_flag",
            "model_ver_prevalid_non_cmplt_cmnt_txt",
            "model_ver_prevalid_method_sid", "model_ver_prevalid_method_code",
            "model_ver_sota_flag", "model_ver_sota_method_name",
            "model_ver_dev_automl_flag",
            "model_ver_dev_automl_library_name",
            "model_ver_dev_automl_baseline_metric_name",
            "model_ver_dev_automl_baseline_metric_val",
            "model_ver_automl_possible_flag", "model_ver_automl_aprvl_flag",
            "model_ver_automl_proj_sid",
            "model_ver_automl_repstry_commit_link_txt",
            "model_ver_automl_repstry_branch_name",
            "model_ver_automl_repstry_commit_sid",
            "model_ver_llm_flag", "model_ver_llm_descr_txt",
            "model_ver_alt_flag", "model_ver_alt_rslt_txt",
            "model_ver_blackbox_flag",
            "model_ver_module_integral_flag", "model_ver_integral_sid",
            "model_ver_calibr_sid", "model_ver_driver_array",
            "model_ver_clone_sid", "model_ver_pltfm_lowcode_name",
            "model_ver_pltfm_lowcode_link_txt",
            "model_ver_calc_freq_name", "model_ver_workflow_val",
            "model_ver_refactoring_plan_dttm",
        ],
        "top": [
            "model_ver_stts_name", "model_ver_signfcnt_ctgry_code",
            "model_ver_signfcnt_lvl_name", "model_ver_supervision_lvl_name",
            "model_ver_irb_flag", "model_ver_method_name",
            "model_ver_task_type_name", "model_ver_data_type_name",
            "model_ver_role_array", "model_ver_dev_src_name",
            "model_ver_dev_sys_name", "model_ver_dev_instr_name",
            "model_ver_dev_not_sample_data_flag",
            "model_ver_repstry_non_link_reason_name",
            "model_ver_proj_feature_flag", "model_ver_prevalid_flag",
            "model_ver_prevalid_rslt_name",
            "model_ver_prevalid_rslt_manual_flag", "model_ver_sota_flag",
            "model_ver_dev_automl_flag", "model_ver_automl_possible_flag",
            "model_ver_automl_aprvl_flag", "model_ver_llm_flag",
            "model_ver_alt_flag", "model_ver_blackbox_flag",
            "model_ver_module_integral_flag",
            "model_ver_pltfm_lowcode_name", "model_ver_calc_freq_name",
        ],
        "rows": [
            "model_ver_sid", "model_sid", "model_ver_num",
            "model_ver_stts_name", "model_ver_signfcnt_ctgry_code",
            "model_ver_method_name", "model_ver_task_type_name",
            "model_ver_data_type_name", "model_ver_dev_src_name",
            "model_ver_dev_sys_name", "model_ver_dev_instr_name",
            "model_ver_dev_start_plan_dttm", "model_ver_dev_start_fact_dttm",
            "model_ver_dev_end_plan_dttm", "model_ver_dev_end_fact_dttm",
            "model_ver_dev_not_sample_data_flag",
            "model_ver_data_mart_link_txt",
            "model_ver_mlstorage_dataset_link_txt",
            "model_ver_repstry_link_txt", "model_ver_repstry_commit_sid",
            "model_ver_repstry_branch_name", "model_ver_dev_report_sid",
            "model_ver_proj_feature_flag", "model_ver_proj_feature_sid",
            "model_ver_prevalid_flag", "model_ver_prevalid_rslt_name",
            "model_ver_sota_flag", "model_ver_dev_automl_flag",
            "model_ver_llm_flag", "model_ver_alt_flag",
            "model_ver_blackbox_flag", "model_ver_module_integral_flag",
            "model_ver_integral_sid", "model_ver_calibr_sid",
            "model_ver_clone_sid",
        ],
    },
    "t_sample_data": {
        "key": "sample_data_sid",
        "columns": [
            "sample_data_sid", "model_ver_sid", "valid_sid",
            "montrg_auto_rslt_sid", "montrg_manual_rslt_sid", "res_sid",
            "sample_data_crtn_dttm", "sample_data_prop_array",
            "sample_data_type_name", "sample_data_stts_name",
            "sample_data_not_metric_calc_flag", "sample_data_addtnl_info_txt",
        ],
        "top": [
            "sample_data_prop_array", "sample_data_type_name",
            "sample_data_stts_name", "sample_data_not_metric_calc_flag",
        ],
        "rows": [
            "sample_data_sid", "model_ver_sid", "valid_sid",
            "montrg_auto_rslt_sid", "montrg_manual_rslt_sid", "res_sid",
            "sample_data_crtn_dttm", "sample_data_prop_array",
            "sample_data_type_name", "sample_data_stts_name",
            "sample_data_not_metric_calc_flag", "sample_data_addtnl_info_txt",
        ],
    },
    "t_metric": {
        "key": "metric_sid",
        "columns": [
            "metric_sid", "sample_data_sid", "metric_crtn_dttm",
            "metric_name", "metric_val", "metric_stts_name",
            "metric_prev_sid", "metric_prev_name",
        ],
        "top": ["metric_name", "metric_stts_name"],
        "rows": [
            "metric_sid", "sample_data_sid", "metric_crtn_dttm",
            "metric_name", "metric_val", "metric_stts_name",
        ],
    },
    "t_model_ver_x_busn_task": {
        "key": ["model_ver_sid", "busn_task_sid"],
        "columns": ["model_ver_sid", "busn_task_sid"],
        "top": [],
        "rows": ["model_ver_sid", "busn_task_sid"],
    },
    "t_busn_task": {
        "key": "busn_task_sid",
        "columns": [
            "busn_task_sid", "busn_task_name", "busn_task_claim_descr_txt",
            "busn_task_crtn_dttm", "busn_task_start_dttm",
            "busn_task_end_dttm", "busn_task_stts_name",
            "busn_task_employer_block_name",
            "busn_task_employer_dprtmt_name", "busn_task_purp_name",
            "busn_task_other_docs_sid",
        ],
        "top": [
            "busn_task_stts_name", "busn_task_employer_block_name",
            "busn_task_employer_dprtmt_name", "busn_task_purp_name",
        ],
        "rows": [
            "busn_task_sid", "busn_task_name", "busn_task_crtn_dttm",
            "busn_task_start_dttm", "busn_task_end_dttm",
            "busn_task_stts_name", "busn_task_employer_block_name",
            "busn_task_employer_dprtmt_name", "busn_task_purp_name",
            "busn_task_other_docs_sid",
        ],
    },
    "t_research": {
        "key": "research_sid",
        "columns": [
            "research_sid", "busn_task_sid", "research_task_name",
            "research_crtn_dttm", "research_start_dttm", "research_end_dttm",
            "research_stts_name", "research_dprtmt_name",
            "research_report_sid", "research_other_docs_sid",
            "research_repstry_name", "research_cmnt_txt",
        ],
        "top": ["research_stts_name", "research_dprtmt_name"],
        "rows": [
            "research_sid", "busn_task_sid", "research_task_name",
            "research_crtn_dttm", "research_start_dttm", "research_end_dttm",
            "research_stts_name", "research_dprtmt_name",
            "research_report_sid", "research_other_docs_sid",
            "research_repstry_name",
        ],
    },
    "t_proj": {
        "key": "proj_sid",
        "columns": [
            "proj_sid", "proj_prnt_sid", "proj_run_sid", "proj_crtn_dttm",
            "proj_upd_last_dttm", "proj_zone_name", "proj_rslt_name",
            "proj_stts_name", "proj_type_name",
        ],
        "top": [
            "proj_zone_name", "proj_rslt_name", "proj_stts_name",
            "proj_type_name",
        ],
        "rows": [
            "proj_sid", "proj_prnt_sid", "proj_run_sid", "proj_crtn_dttm",
            "proj_upd_last_dttm", "proj_zone_name", "proj_rslt_name",
            "proj_stts_name", "proj_type_name",
        ],
    },
    "t_proj_rslt_rb_corp": {
        "key": "proj_rslt_sid",
        "columns": [
            "proj_rslt_sid", "model_ver_sid", "valid_sid", "busn_proc_sid",
            "proj_sid", "proj_run_sid", "proj_rslt_node_sid",
            "proj_run_dttm", "proj_rslt_name", "proj_type_name",
        ],
        "top": ["proj_rslt_name", "proj_type_name"],
        "rows": [
            "proj_rslt_sid", "model_ver_sid", "valid_sid", "busn_proc_sid",
            "proj_sid", "proj_run_sid", "proj_run_dttm",
            "proj_rslt_name", "proj_type_name",
        ],
    },
    "t_proj_test_rb_corp": {
        "key": "proj_test_sid",
        "columns": [
            "proj_test_sid", "model_ver_sid", "proj_sid", "proj_run_sid",
            "proj_test_node_sid", "proj_test_dttm", "proj_test_name",
            "proj_test_sample_name", "proj_test_metric_name",
            "proj_test_metric_val", "proj_test_feature_name",
        ],
        "top": [
            "proj_test_name", "proj_test_sample_name",
            "proj_test_metric_name",
        ],
        "rows": [
            "proj_test_sid", "model_ver_sid", "proj_sid", "proj_run_sid",
            "proj_test_dttm", "proj_test_name", "proj_test_sample_name",
            "proj_test_metric_name", "proj_test_metric_val",
        ],
    },
    "t_proj_test_rslt_eco": {
        "key": "proj_test_sid",
        "columns": [
            "proj_test_sid", "model_ver_sid", "busn_proc_sid",
            "proj_sid", "proj_run_sid", "proj_test_node_sid",
            "proj_test_dttm", "proj_test_name", "proj_rslt_name",
            "proj_test_metric_val", "proj_test_metric_cmnt_txt",
        ],
        "top": ["proj_test_name", "proj_rslt_name"],
        "rows": [
            "proj_test_sid", "model_ver_sid", "busn_proc_sid",
            "proj_sid", "proj_run_sid", "proj_test_dttm",
            "proj_test_name", "proj_rslt_name", "proj_test_metric_val",
        ],
    },
    "t_proj_alt_test_anlt_dtl": {
        "key": "proj_sid",
        "columns": [
            "proj_sid", "model_ver_sid",
            "proj_test_metric_ml_task_val", "proj_test_metric_key_val",
            "proj_test_metric_method_val", "proj_test_metric_baseline_val",
            "proj_test_metric_alt_val", "proj_test_metric_stat_signfcnt_val",
            "proj_test_metric_prevalid_proj_val",
            "proj_test_metric_prevalid_method_val",
            "proj_test_metric_model_auto_rslt_val",
            "proj_test_metric_valid_rslt_val",
            "proj_test_metric_stat_signfcnt_rslt_val",
            "proj_test_metric_valid_auto_link_val",
            "proj_test_metric_valid_auto_rslt_val",
            "proj_test_metric_valid_auto_stat_signfcnt_rslt_val",
            "proj_test_metric_final_val",
            "proj_test_metric_cur_sample_name",
            "proj_test_metric_sample_name",
        ],
        "top": [
            "proj_test_metric_ml_task_val", "proj_test_metric_key_val",
            "proj_test_metric_method_val",
            "proj_test_metric_stat_signfcnt_val",
            "proj_test_metric_model_auto_rslt_val",
            "proj_test_metric_stat_signfcnt_rslt_val",
            "proj_test_metric_final_val",
            "proj_test_metric_cur_sample_name",
            "proj_test_metric_sample_name",
        ],
        "rows": [
            "proj_sid", "model_ver_sid",
            "proj_test_metric_ml_task_val", "proj_test_metric_key_val",
            "proj_test_metric_method_val", "proj_test_metric_baseline_val",
            "proj_test_metric_alt_val", "proj_test_metric_stat_signfcnt_val",
            "proj_test_metric_model_auto_rslt_val",
            "proj_test_metric_stat_signfcnt_rslt_val",
            "proj_test_metric_final_val",
            "proj_test_metric_cur_sample_name",
            "proj_test_metric_sample_name",
        ],
    },
    "t_ent_param_chg": {
        "key": "ent_param_chg_sid",
        "columns": [
            "ent_param_chg_sid", "ent_sid", "ent_type_name",
            "ent_param_chg_name", "ent_param_chg_type_code",
            "ent_param_chg_prev_val", "ent_param_chg_val",
            "start_dttm", "end_dttm",
        ],
        "top": [
            "ent_type_name", "ent_param_chg_name",
            "ent_param_chg_type_code",
        ],
        "rows": [
            "ent_param_chg_sid", "ent_sid", "ent_type_name",
            "ent_param_chg_name", "ent_param_chg_type_code",
            "ent_param_chg_prev_val", "ent_param_chg_val",
            "start_dttm", "end_dttm",
        ],
    },
    "t_model_ver_anlt_dtl": {
        "key": "model_ver_sid",
        "columns": [
            "model_ver_sid", "model_sid", "model_ver_num",
            "model_ver_stts_name", "model_ver_stts_dttm",
            "model_ver_signfcnt_ctgry_code", "model_ver_method_name",
            "model_ver_task_type_name", "model_ver_data_type_name",
            "model_ver_dev_instr_name", "model_ver_run_instr_name",
            "model_ver_dev_src_name", "model_ver_dev_sys_name",
            "model_ver_dev_end_fact_dttm", "model_ver_dev_end_flag",
            "model_ver_dev_time_anomaly_flag", "model_ver_dev_time_grp_name",
            "model_ver_stg_dev_time_qty", "model_ver_initr_jira_task_num",
            "model_ver_busn_proc_code", "model_ver_ab_test_flag",
            "model_ver_alt_flag", "model_ver_alt_improve_flag",
            "model_ver_alt_simplify_flag", "model_ver_irb_flag",
            "model_ver_llm_flag", "model_ver_proj_feature_sid",
            "model_ver_prevalid_rslt_name",
        ],
        "top": [
            "model_ver_stts_name", "model_ver_signfcnt_ctgry_code",
            "model_ver_method_name", "model_ver_task_type_name",
            "model_ver_data_type_name", "model_ver_dev_instr_name",
            "model_ver_run_instr_name", "model_ver_dev_src_name",
            "model_ver_dev_sys_name", "model_ver_dev_end_flag",
            "model_ver_dev_time_anomaly_flag", "model_ver_dev_time_grp_name",
            "model_ver_ab_test_flag", "model_ver_alt_flag",
            "model_ver_alt_improve_flag", "model_ver_alt_simplify_flag",
            "model_ver_irb_flag", "model_ver_llm_flag",
            "model_ver_prevalid_rslt_name",
        ],
        "rows": [
            "model_ver_sid", "model_sid", "model_ver_num",
            "model_ver_stts_name", "model_ver_stts_dttm",
            "model_ver_signfcnt_ctgry_code", "model_ver_method_name",
            "model_ver_task_type_name", "model_ver_data_type_name",
            "model_ver_dev_instr_name", "model_ver_run_instr_name",
            "model_ver_dev_src_name", "model_ver_dev_sys_name",
            "model_ver_dev_end_fact_dttm", "model_ver_dev_end_flag",
            "model_ver_dev_time_anomaly_flag", "model_ver_dev_time_grp_name",
            "model_ver_stg_dev_time_qty", "model_ver_initr_jira_task_num",
            "model_ver_busn_proc_code", "model_ver_ab_test_flag",
            "model_ver_alt_flag", "model_ver_alt_improve_flag",
            "model_ver_alt_simplify_flag", "model_ver_irb_flag",
            "model_ver_llm_flag", "model_ver_proj_feature_sid",
            "model_ver_prevalid_rslt_name",
        ],
    },
}

PII_COLUMN_RE = re.compile(
    r"(^|_)(usr|user|cont|contact|lead|tlds|cds)(_|$)",
    flags=re.IGNORECASE,
)
MASK_VALUE_RE = re.compile(
    r"(link|path|repstry|repository|dataset|cmnt|comment|descr|_txt$|ent_param_chg_(prev_)?val$)",
    flags=re.IGNORECASE,
)
DATE_COLUMN_RE = re.compile(r"(_dt|_dttm|_date)$", flags=re.IGNORECASE)


In [ ]:
OUTPUT_SCHEMA = T.StructType([
    T.StructField("source_table", T.StringType(), False),
    T.StructField("record_type", T.StringType(), False),
    T.StructField("column_name", T.StringType(), True),
    T.StructField("data_type", T.StringType(), True),
    T.StructField("sample_rank", T.IntegerType(), True),
    T.StructField("source_row_count", T.LongType(), True),
    T.StructField("profiled_row_count", T.LongType(), True),
    T.StructField("approx_distinct_key_count", T.LongType(), True),
    T.StructField("possible_duplicate_key_rows", T.LongType(), True),
    T.StructField("non_missing_count", T.LongType(), True),
    T.StructField("basic_missing_count", T.LongType(), True),
    T.StructField("placeholder_count", T.LongType(), True),
    T.StructField("missing_pct", T.DoubleType(), True),
    T.StructField("placeholder_pct", T.DoubleType(), True),
    T.StructField("approx_distinct_count", T.LongType(), True),
    T.StructField("numeric_parseable_count", T.LongType(), True),
    T.StructField("numeric_parseable_pct", T.DoubleType(), True),
    T.StructField("valid_date_count", T.LongType(), True),
    T.StructField("invalid_date_count", T.LongType(), True),
    T.StructField("value_count", T.LongType(), True),
    T.StructField("value_pct", T.DoubleType(), True),
    T.StructField("min_value", T.StringType(), True),
    T.StructField("max_value", T.StringType(), True),
    T.StructField("sample_value", T.StringType(), True),
    T.StructField("row_json", T.StringType(), True),
])

OUTPUT_FIELDS = [field.name for field in OUTPUT_SCHEMA]


def empty_record(table: str, record_type: str) -> Dict[str, object]:
    row = {name: None for name in OUTPUT_FIELDS}
    row["source_table"] = table
    row["record_type"] = record_type
    return row


def parse_timestamp(column):
    value = column.cast("string")
    return F.coalesce(
        column.cast("timestamp"),
        F.to_timestamp(value, "dd.MM.yyyy H:mm:ss"),
        F.to_timestamp(value, "dd.MM.yyyy HH:mm:ss"),
        F.to_timestamp(value, "yyyy-MM-dd HH:mm:ss.SSSSSS"),
        F.to_timestamp(value, "yyyy-MM-dd HH:mm:ss"),
        F.to_timestamp(value, "yyyy-MM-dd"),
    )


def basic_missing(column):
    value = F.lower(F.trim(column.cast("string")))
    return column.isNull() | value.isin("", "null", "none")


def placeholder(column):
    value = F.lower(F.trim(column.cast("string")))
    return value.isin("-", "нет", "n/a", "na", "$not_applicable$", "not_applicable", "not applicable", "не применимо")


def load_actual(table: str, requested: List[str]):
    full_name = f"{SRC_DB}.{table}"
    if not spark.catalog.tableExists(full_name):
        return None, requested, {}

    raw = spark.table(full_name)
    actual_names = {name.lower(): name for name in raw.columns}
    missing = [name for name in requested if name.lower() not in actual_names]

    if "ctl_action" in actual_names:
        ctl = F.col(actual_names["ctl_action"])
        raw = raw.filter(
            ctl.isNull() | (F.upper(F.trim(ctl.cast("string"))) != "D")
        )
    if "start_dt" in actual_names:
        start = parse_timestamp(F.col(actual_names["start_dt"]))
        raw = raw.filter(start.isNull() | (F.to_date(start) <= F.current_date()))
    if "end_dt" in actual_names:
        end = parse_timestamp(F.col(actual_names["end_dt"]))
        raw = raw.filter(end.isNull() | (F.to_date(end) >= F.current_date()))

    selected = [
        F.col(actual_names[name.lower()]).alias(name.lower())
        for name in requested
        if name.lower() in actual_names and not PII_COLUMN_RE.search(name)
    ]
    if not selected:
        return raw.select(), missing, {}
    df = raw.select(*selected)
    return df, missing, dict(df.dtypes)


def profile_table(table: str, spec: Dict[str, object]) -> List[Dict[str, object]]:
    requested = [str(c).lower() for c in spec["columns"]]
    key_spec = spec["key"]
    keys = (
        [str(value).lower() for value in key_spec]
        if isinstance(key_spec, (list, tuple))
        else [str(key_spec).lower()]
    )
    df, missing_columns, data_types = load_actual(table, requested)
    rows: List[Dict[str, object]] = []

    if df is None:
        row = empty_record(table, "TABLE_MISSING")
        row["row_json"] = f"{SRC_DB}.{table}"
        rows.append(row)
        return rows

    for column in missing_columns:
        row = empty_record(table, "SCHEMA_MISSING")
        row["column_name"] = column
        rows.append(row)

    counts = df.agg(
        F.count(F.lit(1)).cast("long").alias("source_rows"),
        (
            F.approx_count_distinct(
                F.xxhash64(*[F.col(key) for key in keys])
            ).cast("long")
            if all(key in df.columns for key in keys)
            else F.lit(None).cast("long")
        ).alias("distinct_keys"),
    ).first()
    source_count = int(counts["source_rows"] or 0)
    distinct_keys = counts["distinct_keys"]
    possible_duplicates = (
        max(source_count - int(distinct_keys), 0)
        if distinct_keys is not None
        else None
    )

    if source_count <= MAX_PROFILE_ROWS:
        sampled = df
    else:
        fraction = min(1.0, 1.25 * MAX_PROFILE_ROWS / source_count)
        sampled = df.sample(False, fraction, SEED).limit(MAX_PROFILE_ROWS)

    sampled = sampled.persist(StorageLevel.MEMORY_AND_DISK)
    profiled_count = sampled.count()

    summary = empty_record(table, "TABLE_SUMMARY")
    summary.update({
        "column_name": "|".join(keys),
        "data_type": "|".join(data_types.get(key, "?") for key in keys),
        "source_row_count": source_count,
        "profiled_row_count": profiled_count,
        "approx_distinct_key_count": (
            int(distinct_keys) if distinct_keys is not None else None
        ),
        "possible_duplicate_key_rows": possible_duplicates,
        "row_json": (
            '{"missing_configured_columns":['
            + ",".join(f'"{c}"' for c in missing_columns)
            + "]}"
        ),
    })
    rows.append(summary)

    if profiled_count == 0:
        sampled.unpersist()
        return rows

    expressions = []
    profile_columns = [
        column for column in sampled.columns if not PII_COLUMN_RE.search(column)
    ]
    for column in profile_columns:
        source = F.col(column)
        missing = basic_missing(source)
        placeholder_flag = (~missing) & placeholder(source)
        non_missing = ~missing
        prefix = f"p__{column}__"

        expressions.extend([
            F.sum(F.when(non_missing, 1).otherwise(0)).cast("long").alias(prefix + "non_missing"),
            F.sum(F.when(missing, 1).otherwise(0)).cast("long").alias(prefix + "missing"),
            F.sum(F.when(placeholder_flag, 1).otherwise(0)).cast("long").alias(prefix + "placeholder"),
            F.approx_count_distinct(
                F.when(non_missing, source.cast("string"))
            ).cast("long").alias(prefix + "distinct"),
            F.sum(
                F.when(non_missing & source.cast("double").isNotNull(), 1).otherwise(0)
            ).cast("long").alias(prefix + "numeric"),
        ])

        if DATE_COLUMN_RE.search(column):
            parsed = parse_timestamp(source)
            valid_date = (
                non_missing
                & (F.to_date(parsed) >= F.lit("2000-01-01").cast("date"))
                & (F.to_date(parsed) < F.lit("2090-01-01").cast("date"))
            )
            expressions.extend([
                F.sum(F.when(valid_date, 1).otherwise(0)).cast("long").alias(prefix + "valid_date"),
                F.sum(F.when(non_missing & ~valid_date, 1).otherwise(0)).cast("long").alias(prefix + "invalid_date"),
            ])
        else:
            expressions.extend([
                F.lit(None).cast("long").alias(prefix + "valid_date"),
                F.lit(None).cast("long").alias(prefix + "invalid_date"),
            ])

        if MASK_VALUE_RE.search(column):
            expressions.extend([
                F.lit(None).cast("string").alias(prefix + "min"),
                F.lit(None).cast("string").alias(prefix + "max"),
            ])
        else:
            normalized = F.when(non_missing, source.cast("string"))
            expressions.extend([
                F.min(normalized).cast("string").alias(prefix + "min"),
                F.max(normalized).cast("string").alias(prefix + "max"),
            ])

    metrics = sampled.agg(*expressions).first().asDict()

    for column in profile_columns:
        prefix = f"p__{column}__"
        non_missing = int(metrics[prefix + "non_missing"] or 0)
        missing_count = int(metrics[prefix + "missing"] or 0)
        placeholder_count = int(metrics[prefix + "placeholder"] or 0)
        numeric_count = int(metrics[prefix + "numeric"] or 0)

        row = empty_record(table, "COLUMN_PROFILE")
        row.update({
            "column_name": column,
            "data_type": data_types.get(column),
            "source_row_count": source_count,
            "profiled_row_count": profiled_count,
            "non_missing_count": non_missing,
            "basic_missing_count": missing_count,
            "placeholder_count": placeholder_count,
            "missing_pct": round(100.0 * missing_count / profiled_count, 6),
            "placeholder_pct": round(100.0 * placeholder_count / profiled_count, 6),
            "approx_distinct_count": int(metrics[prefix + "distinct"] or 0),
            "numeric_parseable_count": numeric_count,
            "numeric_parseable_pct": (
                round(100.0 * numeric_count / non_missing, 6)
                if non_missing else None
            ),
            "valid_date_count": metrics[prefix + "valid_date"],
            "invalid_date_count": metrics[prefix + "invalid_date"],
            "min_value": metrics[prefix + "min"],
            "max_value": metrics[prefix + "max"],
        })
        rows.append(row)

    for column in spec["top"]:
        column = str(column).lower()
        if column not in sampled.columns or PII_COLUMN_RE.search(column):
            continue
        source = F.col(column)
        normalized = (
            F.when(basic_missing(source), F.lit("$NULL$"))
            .when(placeholder(source), F.lit("$PLACEHOLDER$"))
            .otherwise(
                F.regexp_replace(F.trim(source.cast("string")), r"\s+", " ")
            )
            .alias("sample_value")
        )
        top_rows = (
            sampled.select(normalized)
            .groupBy("sample_value")
            .count()
            .orderBy(F.desc("count"), F.asc("sample_value"))
            .limit(TOP_VALUES_PER_COLUMN)
            .collect()
        )
        for rank, value_row in enumerate(top_rows, start=1):
            value_count = int(value_row["count"])
            row = empty_record(table, "TOP_VALUE")
            row.update({
                "column_name": column,
                "data_type": data_types.get(column),
                "sample_rank": rank,
                "source_row_count": source_count,
                "profiled_row_count": profiled_count,
                "value_count": value_count,
                "value_pct": round(100.0 * value_count / profiled_count, 6),
                "sample_value": value_row["sample_value"],
            })
            rows.append(row)

    row_columns = [
        str(column).lower()
        for column in spec["rows"]
        if str(column).lower() in sampled.columns
        and not PII_COLUMN_RE.search(str(column))
    ]
    if row_columns:
        payload_columns = []
        for column in row_columns:
            source = F.col(column)
            if MASK_VALUE_RE.search(column):
                payload = F.when(
                    basic_missing(source), F.lit(None).cast("string")
                ).otherwise(
                    F.concat(
                        F.lit("sha256:"),
                        F.sha2(F.trim(source.cast("string")), 256),
                    )
                )
            else:
                payload = source.cast("string")
            payload_columns.append(payload.alias(column))

        sample_json = (
            sampled.select(F.to_json(F.struct(*payload_columns)).alias("row_json"))
            .dropDuplicates(["row_json"])
            .orderBy(F.xxhash64("row_json"))
            .limit(ROW_SAMPLES_PER_TABLE)
            .collect()
        )
        for rank, value_row in enumerate(sample_json, start=1):
            row = empty_record(table, "ROW_SAMPLE")
            row.update({
                "sample_rank": rank,
                "source_row_count": source_count,
                "profiled_row_count": profiled_count,
                "row_json": value_row["row_json"],
            })
            rows.append(row)

    sampled.unpersist()
    return rows


In [ ]:
all_rows: List[Dict[str, object]] = []

for index, (table, spec) in enumerate(TABLES.items(), start=1):
    print(f"[{index:02d}/{len(TABLES):02d}] profiling {table}")
    try:
        table_rows = profile_table(table, spec)
        all_rows.extend(table_rows)
        print(f"  output rows: {len(table_rows)}")
    except Exception as exc:
        error = empty_record(table, "TABLE_ERROR")
        error["row_json"] = f"{type(exc).__name__}: {str(exc)[:2000]}"
        all_rows.append(error)
        print(f"  ERROR: {error['row_json']}")

if not all_rows:
    raise RuntimeError("Профиль не сформирован")

profile_df = (
    spark.createDataFrame(all_rows, schema=OUTPUT_SCHEMA)
    .withColumn("as_of_dt", F.current_date())
    .persist(StorageLevel.MEMORY_AND_DISK)
)

if profile_df.filter(
    F.lower(F.col("column_name")).rlike(
        r"(^|_)(usr|user|cont|contact|lead|tlds|cds)(_|$)"
    )
).limit(1).count():
    raise RuntimeError("В профиль попала потенциально персональная колонка")

expected_tables = len(TABLES)
actual_tables = profile_df.select("source_table").distinct().count()
if actual_tables != expected_tables:
    raise RuntimeError(
        f"Ожидалось таблиц: {expected_tables}; представлено в профиле: {actual_tables}"
    )

print("\n=== TABLE SUMMARY ===")
profile_df.filter(
    F.col("record_type").isin("TABLE_SUMMARY", "TABLE_MISSING", "TABLE_ERROR")
).orderBy("source_table").show(100, truncate=False)

print("\n=== SCHEMA GAPS ===")
profile_df.filter(
    F.col("record_type") == "SCHEMA_MISSING"
).orderBy("source_table", "column_name").show(300, truncate=False)

print("\n=== MOST INCOMPLETE COLUMNS ===")
profile_df.filter(
    F.col("record_type") == "COLUMN_PROFILE"
).orderBy(
    F.desc("missing_pct"), F.desc("placeholder_pct")
).select(
    "source_table", "column_name", "data_type",
    "profiled_row_count", "missing_pct", "placeholder_pct",
    "approx_distinct_count", "numeric_parseable_pct",
    "valid_date_count", "invalid_date_count",
).show(100, truncate=False)

print("\n=== TOP VALUES PREVIEW ===")
profile_df.filter(
    F.col("record_type") == "TOP_VALUE"
).orderBy(
    "source_table", "column_name", "sample_rank"
).select(
    "source_table", "column_name", "sample_rank",
    "sample_value", "value_count", "value_pct",
).show(200, truncate=False)

(
    profile_df.repartition(1)
    .write.mode("overwrite")
    .format("parquet")
    .saveAsTable(OUT_FULL_NAME)
)

print(f"\nГотово: {OUT_FULL_NAME}")
print(f"Строк в профиле: {profile_df.count()}")

profile_df.unpersist()
